---
image: example.gif
pub-info:
    abstract: |
        A simple queueing model built with Ciw rather than SimPy, showing how to convert Ciw's own
        simulation records into a vidigi-compatible event log. Based on a published reproducible Ciw
        model example from Monks, Harper & Heather (2023).
execute: 
  enabled: true
---

# A Simple Ciw Model

Note that this example is written using ciw 2.x

It will not run with 3.x - but could theoretically be adapted to do so

The 'logs' object is the result of running

`sim_engine.get_all_records()`

However, note that while we run multiple replications, we only pass the records for a single replication to the 
`event_log_from_ciw_recs` function. 

---

The underlying model code is from Monks, T., Harper, A., & Heather, A. (2023). Towards Sharing Tools, Artefacts, and Reproducible Simulation: a ciw model example (v1.0.1). Zenodo. https://doi.org/10.5281/zenodo.10051494

See here for the adaptation embedded within that repo: [https://github.com/Bergam0t/ciw-example-animation/tree/main](https://github.com/Bergam0t/ciw-example-animation/tree/main)

---

In SimPy models, we have to manually add our event logs at various points. However, for Ciw models, we instead can make use of the `event_log_from_ciw_recs` helper function from `vidigi.utils` to automatically reshape the logs ciw generates into the correct format for vidigi to work with. 

Let's start by running the model and viewing the logs ciw outputs. 

In [1]:
import pandas as pd
# Import the wrapper objects for model interaction.
from ex_4_ciw_model import Experiment, multiple_replications
from vidigi.ciw import event_log_from_ciw_recs
from vidigi.animation import animate_activity_log
from vidigi.utils import create_event_position_df, EventPosition
import os

ModuleNotFoundError: No module named 'vidigi'

In [ ]:
#| echo: false
import plotly.io as pio
pio.renderers.default = "notebook"

In [ ]:
#| echo: false
#| output: asis
# Path to the external Python script
file_path = "ex_4_ciw_model.py"

# Read the file content
if os.path.exists(file_path):
    with open(file_path, "r") as f:
        code_content = f.read()
else:
    code_content = "File not found."
with open(file_path, "r") as f:
    code_content = f.read()

# Print the Quarto `{details}` block for collapsible output
print(f"""
:::{{.callout-note collapse="true"}}
### View Imported Code for the ciw model

```python
{code_content}
```

:::

""")

In [ ]:
N_OPERATORS = 18
N_NURSES = 9
RESULTS_COLLECTION_PERIOD = 1000

user_experiment = Experiment(n_operators=N_OPERATORS,
                             n_nurses=N_NURSES,
                             chance_callback=0.4)

# run multiple replications
results, logs = multiple_replications(user_experiment, n_reps=10)


While we've done multiple replications, for the purpose of the animation we want only a single set of logs, so we will extract those from the logs variable we created. 

In [ ]:
# the 'logs' object contains a list, where each entry is the recs object for that run
logs_run_1 = logs[0]

print(len(logs_run_1))

Let's look at the first row of our result. What do we have? 

In [ ]:
logs[0][0]

Let's print all of the outputs for a single individual.

In [ ]:
[print(log) for log in logs_run_1 if log.id_number==500]

It looks like we get one entry per node that is visited. 

As mentioned, we can make use of the `event_log_from_ciw_recs` helper function from `vidigi.utils` to automatically reshape ciw logs into the correct format for vidigi to work with. 

For each node, we pass in an appropriate name. Vidigi will use these and append '_begins' and '_ends', as well as calculating arrivals and departures from the model and creating resource IDs to allow it to correctly show the utilisation of a resource at each step. 

In [ ]:
# let's now try turning this into an event log
event_log_test = event_log_from_ciw_recs(logs_run_1, node_name_list=["operator", "nurse"])

event_log_test.head(25)

Now we need to create a suitable class to pass in the resource numbers to the animation function.

In [ ]:
# Create a suitable class to pass in the resource numbers to the animation function
class model_params():
    def __init__(self):
        self.n_operators = N_OPERATORS
        self.n_nurses = N_NURSES

params = model_params()

print(f"Number of operators: {params.n_operators}")
print(f"Number of nurses: {params.n_nurses}")

Like with SimPy, we need to tell vidigi where to put each step on our plot. We will refer to the names we used - so as we named our nodes 'operator' and 'nurse', we will want

- arrival
- operator_wait_begins (to show queueing for the operator)
- operator_begins (to show resource use of the operator)
- nurse_wait_begins (to show queuing for the nurse after finishing being seen by the operator)
- nurse_begins (to show resource use of the nurse)

For the _begins steps, which relat to resource use, we will also pass in a name that relates to the number of resources we need, which we defined in our model_params class above.

So, for the operator_begins step, for example, we tell ut to look for n_operators, whch is one of the parameters in our model_params class. We pass the params class into the animation function. 

In [ ]:
# Create required event_position_df for vidigi animation
event_position_df = create_event_position_df([
    EventPosition(event='operator_wait_begins', x=205, y=270, label="Waiting for Operator"),
    EventPosition(event='operator_begins', x=210, y=210, resource='n_operators', label="Speaking to Operator"),
    EventPosition(event= 'nurse_wait_begins', x=205, y=110, label="Waiting for Nurse"),
    EventPosition(event= 'nurse_begins', x=210, y=50, resource='n_nurses', label="Speaking to Nurse"),
    EventPosition(event= 'depart', x=270, y=10, label="Exit")
])

event_position_df

Finally, we can create the animation. 

In [ ]:
# Create animation
params = model_params()

animate_activity_log(
        event_log=event_log_test,
        event_position_df=event_position_df,
        scenario=model_params(),
        debug_mode=True,
        setup_mode=False,
        every_x_time_units=1,
        include_play_button=True,
        entity_icon_size=20,
        gap_between_entities=8,
        gap_between_queue_rows=25,
        plotly_height=700,
        frame_duration=200,
        plotly_width=1200,
        override_x_max=300,
        override_y_max=300,
        limit_duration=RESULTS_COLLECTION_PERIOD,
        wrap_queues_at=25,
        wrap_resources_at=50,
        step_snapshot_max=75,
        time_display_units="dhm",
        display_stage_labels=True,
    )
